<a href="https://colab.research.google.com/github/wjdwogns2873-web/deep-learning-study/blob/main/Kaggle_Study/08_Dog_breed_identification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 1. 구글 드라이브 연결 (로그인 팝업이 뜨면 확인만 눌러주세요)
from google.colab import drive
drive.mount('/content/drive')

# 2. 구글 드라이브에 저장해둔 열쇠(access_token)를 코랩 보안 폴더로 자동 복사
!mkdir -p ~/.kaggle
!cp /content/drive/MyDrive/Kaggle/access_token ~/.kaggle/
!chmod 600 ~/.kaggle/access_token

# 3. 캐글 API로 MNIST 데이터 초고속 다운로드 및 압축 해제
!pip install -q kaggle

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!kaggle competitions download -c dog-breed-identification

100% 691M/691M [00:07<00:00, 96.8MB/s]



In [ ]:
# -q(quiet): 이걸 안 붙이면 수만 장의 강아지 이미지 파일명이 출력창에 끝없이 도배됩니다.
# -d ./dog_breed: 압축이 풀린 파일들이 현재 경로 아래 dog_breed 라는 새 디렉토리로 예쁘게 모이도록 지정합니다.
!unzip -q dog-breed-identification.zip -d ./dog_breed

!ls -l ./dog_breed

total 26332
-rw-r--r-- 1 root root   482063 Dec 11  2019 labels.csv
-rw-r--r-- 1 root root 25200295 Dec 11  2019 sample_submission.csv
drwxr-xr-x 2 root root   643072 May 27 16:16 test
drwxr-xr-x 2 root root   626688 May 27 16:16 train


In [ ]:
import torch
import os

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)
if device.type == 'cuda':
    print(f"GPU 모델명: {torch.cuda.get_device_name(0)}")

os.makedirs('./kaggle_data', exist_ok=True)

cuda
GPU 모델명: Tesla T4


In [ ]:
import pandas as pd
from PIL import Image
from torch.utils.data import Dataset
from torchvision import transforms

class DogBreedDataset(Dataset):
    def __init__(self, df, img_dir, transform=None):
        # df: 이미지 파일명과 라벨이 적힌 판다스 데이터프레임
        # img_dir: 실제 이미지 파일들이 저장된 폴더 경로
        # transform: 이미지에 적용할 전처리/증강 파이프라인
        self.df = df
        self.img_dir = img_dir
        self.transform = transform
    def __len__(self):
        # 데이터셋의 총 샘플 개수를 반환
        return len(self.df)
    def __getitem__(self, idx):
        # 이미지 경로 결합 및 로드
        img_name = self.df.iloc[idx, 0] # 첫 번째 컬럼: 파일명 (예: image_0001.jpg)
        img_path = os.path.join(self.img_dir, img_name)
        image = Image.open(img_path).convert('RGB') # PIL을 이용해 이미지 로드

        # 2. 라벨 가져오기
        label = self.df.iloc[idx, 1] # 두 번째 컬럼: 정답 클래스 (정수형)

        # 3. 전처리(Transform) 적용
        if self.transform:
            image = self.transform(image)

        return image, label

`iloc`은 파이썬 Pandas 라이브러리에서 데이터프레임의 행이나 열을 숫자 인덱스(위치) 기준으로 선택할 때 사용하는 함수입니다.

`df.iloc[행_인덱스, 열_인덱스]`

`img_name = self.df.iloc[idx, 0]`에서 원하는 idx를 가져오면, 해당 행(idx)에 해당하는 img_name(파일명)을 가져올 수 있습니다.

In [ ]:
import numpy as np

train_df = pd.read_csv('./dog_breed/labels.csv')
print(len(train_df))
print(train_df.head())

# 1. 문자열 라벨을 고유한 정수(0~119)로 매핑
unique_breeds = train_df['breed'].unique()
unique_breeds.sort()

breed_to_idx = {breed: idx for idx, breed in enumerate(unique_breeds)}
idx_to_breed = {idx: breed for idx, breed in enumerate(unique_breeds)} # 나중에 복원용

# 2. 데이터프레임에 정수형 라벨 컬럼 추가
train_df['label_idx'] = train_df['breed'].map(breed_to_idx)

# 3. 파일명 뒤에 .jpg 확장자 붙여주기
train_df['img_file'] = train_df['id'] + '.jpg'

print(train_df[['img_file', 'breed', 'label_idx']].head())

10222
                                 id             breed
0  000bec180eb18c7604dcecc8fe0dba07       boston_bull
1  001513dfcb2ffafc82cccf4d8bbaba97             dingo
2  001cdf01b096e06d78e9e5112d419397          pekinese
3  00214f311d5d2247d5dfe4fe24b2303d          bluetick
4  0021f9ceb3235effd7fcde7f7538ed62  golden_retriever
                               img_file             breed  label_idx
0  000bec180eb18c7604dcecc8fe0dba07.jpg       boston_bull         19
1  001513dfcb2ffafc82cccf4d8bbaba97.jpg             dingo         37
2  001cdf01b096e06d78e9e5112d419397.jpg          pekinese         85
3  00214f311d5d2247d5dfe4fe24b2303d.jpg          bluetick         15
4  0021f9ceb3235effd7fcde7f7538ed62.jpg  golden_retriever         49


`unique()`: 중복 제거하기

`labels.csv` 파일에는 수천마리의 강아지 데이터가 있어서 같은 품종 이름이 반복됩니다.

`unique()`를 함으로써 중복이 모두 사라지고, 딱 120개의 고유한 품좀 이름만 리스트(배열) 형태로 남습니다.

breed_to_idx: 문자열을 숫자로 매핑하는 사전(Dictionary)

아래와 같은 딕셔너리 자료형이 만들어집니다.
```
{
    'affenpinscher': 0,
    'afghan_hound': 1,
    'african_hunting_dog': 2,
    ...
    'yorkshire_terrier': 119
}
```
`map()`: 딕셔너리를 이용해 통째로 바꾸기

In [ ]:
# 궁금해서 찍어본 것들
print(type(unique_breeds))
print(len(unique_breeds))
print(unique_breeds.shape)

<class 'numpy.ndarray'>
120
(120,)


In [ ]:
class DogBreedDataset(Dataset):
    def __init__(self, df, img_dir, transform=None):
        self.df = df
        self.img_dir = img_dir
        self.transform = transform
    def __len__(self):
        return len(self.df)
    def __getitem__(self, idx):
        img_name = self.df.iloc[idx]['img_file']
        img_path = os.path.join(self.img_dir, img_name)
        image = Image.open(img_path).convert('RGB')

        label = self.df.iloc[idx]['label_idx']

        if self.transform:
            image = self.transform(image)

        return image, label

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], # ImageNet 데이터셋 기준 정규화
                         std=[0.229, 0.224, 0.225])
])

# 데이터셋 객체 생성
train_img_dir = './dog_breed/train'
full_dataset = DogBreedDataset(df=train_df, img_dir=train_img_dir, transform=train_transform)
print(len(full_dataset))

10222


In [ ]:
from torch.utils.data import DataLoader, random_split

# 1. Train / Validation Split (8:2)
train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size

train_dataset, val_dataset = random_split(full_dataset, [train_size, val_size])

BATCH_SIZE = 64

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

images, labels = next(iter(train_loader))
print(images.shape, labels.shape)

torch.Size([64, 3, 224, 224]) torch.Size([64])


그래서

`train_dataset, val_dataset = random_split(full_dataset, [train_size, val_size])`

이 코드도 다음과 같이 간단하게 바꿀 수 있다.

`train_dataset, val_dataset = random_split(full_dataset, [0.8, 0.2])`

In [ ]:
# 궁금해서 구글링한 코드
# random_split은 PyTorch에서 전체 데이터셋을 무작위로 복수의 하위 데이터셋으로 분할할 때 사용하는 함수입니다.

import torch
from torch.utils.data import random_split

# 전체 데이터셋 크기가 10,000개라고 가정
total_dataset = train_dataset()

# 1. 개수(정수) 기준으로 분할하기 (8,000개 / 2,000개)
train_set, val_set = random_split(total_dataset, [8000, 2000])

# 2. 비율 기준으로 분할하기 (80% / 10% / 10%)
train_set, val_set, test_set = random_split(total_dataset, [0.8, 0.1, 0.1])

NameError: name 'train_dataset' is not defined

In [ ]:
import torch.nn as nn
from torchvision import models

# 1. ImageNet 데이터로 사전 학습된 ResNet-50 모델 로드
weights = models.ResNet50_Weights.DEFAULT
model = models.resnet50(weights=weights)

# 2. 모델의 마지막 출력 레이어(fc layer) 구조 확인 및 개조
# ResNet50의 마지막 레이어 이름은 fc입니다. 원래는 1000개의 클래스 분류용으로 되어 있습니다.
in_features = model.fc.in_features

# 우리의 목적에 맞게 120개 클래스를 출력하도록 레이어 교체
model.fc = nn.Linear(in_features, 120)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)

print(device)
if device.type == 'cuda':
    print(f"GPU 모델명: {torch.cuda.get_device_name(0)}")

Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 127MB/s]


cuda
GPU 모델명: Tesla T4


In [ ]:
import torch.optim as optim

criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(model.parameters(), lr=1e-4)

In [ ]:
import time

epochs = 3

for epoch in range(epochs):
    start_time = time.time()

    model.train()
    train_loss, train_correct, total_train = 0.0, 0.0, 0.0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        train_loss += loss.item() * images.size(0)
        _, predicted = torch.max(outputs, 1)
        train_correct += (predicted == labels).sum().item()
        total_train += labels.size(0)

    epoch_train_loss = train_loss / total_train
    epoch_train_acc = train_correct / total_train * 100

    model.eval()
    val_loss, val_correct, total_val = 0.0, 0.0, 0.0

    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)
            val_loss += loss.item() * images.size(0)
            _, predicted = torch.max(outputs, 1)
            val_correct += (predicted == labels).sum().item()
            total_val += labels.size(0)

    epoch_val_loss = val_loss / total_val
    epoch_val_acc = val_correct / total_val * 100

    end_time = time.time()
    epoch_mins, epoch_secs = divmod(end_time - start_time, 60)

    print(f"Epoch [{epoch+1}/{epochs}] | Time: {int(epoch_mins)}m {int(epoch_secs)}s")
    print(f"Train Loss: {epoch_train_loss:.4f} | Train Acc: {epoch_train_acc:.2f}")
    print(f"Val Loss: {epoch_val_loss:.4f} | Val Acc: {epoch_val_acc:.2f}\n")

Epoch [1/3] | Time: 1m 37s
Train Loss: 2.7741 | Train Acc: 51.72
Val Loss: 1.0683 | Val Acc: 77.70

Epoch [2/3] | Time: 1m 37s
Train Loss: 0.6537 | Train Acc: 84.82
Val Loss: 0.5922 | Val Acc: 82.59

Epoch [3/3] | Time: 1m 37s
Train Loss: 0.3115 | Train Acc: 91.90
Val Loss: 0.5558 | Val Acc: 82.89



In [ ]:
# divmod(a, b) 함수
# 두 개의 숫자 인자를 받아 나눗셈의 몫과 나머지를 동시에 구하고, 이를 튜플 형태로 반환합니다.
# (a // b, a % b)와 동일한 결과를 냅니다.
# 몫과 나머지를 한 번에 처리하므로 코드가 깔끔해집니다.

result = divmod(10, 3)
print(result)

quotient, remainder = divmod(10, 3)
print(f"몫: {quotient}, 나머지: {remainder}")

(3, 1)
몫: 3, 나머지: 1


In [ ]:
import time

current_time = time.time()
print(current_time)

# 코드 실행 시간 측정하기
start_time = time.time()

time.sleep(1.5)

end_time = time.time()

elapsed_time = end_time - start_time
print(f"코드 실행 시간: {elapsed_time:.4f}초")

1779967270.3114371
코드 실행 시간: 1.5009초
